In [44]:
import numpy as np
from collections import Counter
import pandas as pd
#from sklearn.model_selection import train_test_split

   


In [45]:
class Node:   
    def __init__(self, feature_idx = None, threshold = None, info_gained = None, left = None, right = None, value = None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.info_gained = info_gained
        self.left = left
        self.right = right
        self.value = value


In [46]:
class Dec_tree:
    def __init__(self, min_samp_split = 10, max_depth = 10, n_features=None):
        self.min_samp_split = min_samp_split
        self.max_depth = max_depth
        self.n_features = n_features

    def build_tree(self,data, curr_depth = 0):
        X,y = data[:, :-1], data[:,-1]
        samples , features = X.shape

        if samples > self.min_samp_split and curr_depth < self.max_depth:
            best_split = self.best_split(data, features)

            if best_split['info_gained'] > 0:
                left_node = self.build_tree(best_split['left_dataset'],curr_depth + 1)
                right_node = self.build_tree(best_split['right_dataset'],curr_depth + 1)

                return Node(best_split['feature_idx'], best_split['threshold'], best_split['info_gained'],left_node,right_node)

        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value = leaf_value)
        
    def best_split(self,data,features):
        if self.n_features is None:
            feature_idxs = np.arange(features)
        else:
            feature_idxs = np.random.choice(features, self.n_features, replace=False)
            
        best_split = {'feature_idx': None, 'threshold':None,'info_gained' : -1, 'left_dataset' : None, 'right_dataset' : None}

        for feature_idx in feature_idxs:
            features_value = data[:,feature_idx]
            
            features_value = np.sort(np.unique(data[:, feature_idx]))
            thresholds = (features_value[:-1] + features_value[1:]) / 2


            for threshold in thresholds:
                left_dataset , right_dataset = self.split(data,feature_idx, threshold)

                if len(left_dataset) and len(right_dataset):
                    parent_y, left_y, right_y = data[:,-1], left_dataset[:,-1],right_dataset[:,-1]

                    info_gained = self.information_gain(parent_y,left_y,right_y)

                    if info_gained > best_split['info_gained']:
                        best_split['feature_idx'] = feature_idx
                        best_split['threshold'] = threshold
                        best_split['info_gained'] = info_gained
                        best_split['left_dataset'] = left_dataset
                        best_split['right_dataset'] = right_dataset

        return best_split
    



    def split(self,data,feature_idx, threshold):
        #left_dataset = np.array([row for row in data if row[feature_idx] <= threshold])
        #right_dataset = np.array([row for row in data if row[feature_idx] > threshold])
        mask = data[:, feature_idx] <= threshold
        left_dataset = data[mask]
        right_dataset = data[~mask]
        return left_dataset, right_dataset    

    def information_gain(self,parent_y,left_y, right_y):
        left_w = len(left_y) / len(parent_y)

        right_w = len(right_y) / len(parent_y)

        info_gain = self.entropy(parent_y) - (left_w*self.entropy(left_y)+right_w*self.entropy(right_y))

        return info_gain
    
    def entropy(self,y):
        entropy = 0

        class_lables = np.unique(y)
        for class_lable in class_lables:
            p = len(y[y==class_lable]) / len(y)
            
            if p > 0:
                entropy += -p * np.log2(p)


        return entropy
    

    def fit(self,X,y,feature_names = None):
        self.feature_names = feature_names
        data = np.concatenate([X,y.reshape(-1,1)],axis=1)
        self.root = self.build_tree(data)

        

    def predict(self,X):
        predict = [self.predict_class(row,self.root) for row in X]
        return predict
    
    def predict_class(self,row,node):
        if node.value != None:
            return node.value

        feature_val = row[node.feature_idx]
        if feature_val <= node.threshold:
            return self.predict_class(row,node.left)
        else:
            return self.predict_class(row,node.right)
        
    def print_tree(self, node=None, indent=""):
        if node is None:
            node = self.root

        if node.value is not None:
            print(indent + "Leaf:", node.value)
            return
        if hasattr(self, "feature_names") and self.feature_names is not None:
            feature_name = self.feature_names[node.feature_idx]
        else:
            feature_name = f"X{node.feature_idx}"


        print(indent + f"[{feature_name} <= {node.threshold}]")
        self.print_tree(node.left, indent + "  ")
        self.print_tree(node.right, indent + "  ")

In [47]:


class random_forest:
    def __init__(self,n_tree = 10 , max_depth = 10,min_samp_split = 2,n_features =None):
        self.n_tree = n_tree
        self.max_depth = max_depth
        self.min_samp_split = min_samp_split
        self.n_features = n_features
        self.trees = []

    def fit(self, X,y, feature_names=None):
        self.trees = []
        n_features = X.shape[1]
        
        if self.n_features is None:
            n_feats = int(np.sqrt(n_features))
        else:
            n_feats = self.n_features

            
        for _ in range(self.n_tree):
            tree = Dec_tree(min_samp_split = self.min_samp_split,
                            max_depth = self.max_depth,
                            n_features = n_feats)

            X_sample, y_sample = self.bootstrap_sample(X,y)
            tree.fit(X_sample, y_sample, feature_names)
            self.trees.append(tree)
    
    def bootstrap_sample(self,X,y):
        n_sample = X .shape[0]
        idxs = np.random.choice(n_sample,n_sample, replace = True)
        return X[idxs], y[idxs]
    
    def most_common_lables(self,y):
        counter = Counter(y)
        most_Common =  counter.most_common(1)[0][0]
        return most_Common
        
    def predict(self,X):
        pred = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(pred,0,1)
        predictions = np.array([self.most_common_lables(pre) for pre in tree_preds])
        return predictions
    





In [48]:
df_train = pd.read_csv('/kaggle/input/dataset-p/train_pca.csv')
df_test = pd.read_csv('/kaggle/input/dataset-p/test_pca.csv')


feature_names_origin = df_test.drop(
    columns=['Heart_Disease_Risk','PC1','PC2']
).columns


feature_names_pca = df_test.drop(
    columns=['Heart_Disease_Risk',
             'Age','Height_cm','Weight_kg','BMI','Systolic_BP',
             'Diastolic_BP','Cholesterol_Total','Cholesterol_HDL',
             'Cholesterol_LDL','Fasting_Blood_Sugar']
).columns



In [49]:
#pca dts

Xp_train = df_train.drop(columns = ['Heart_Disease_Risk',
'Age','Height_cm','Weight_kg','BMI','Systolic_BP',
    'Diastolic_BP','Cholesterol_Total','Cholesterol_HDL',
    'Cholesterol_LDL','Fasting_Blood_Sugar']).values


Yp_train = df_train['Heart_Disease_Risk'].values



Xp_test = df_test.drop(columns = ['Heart_Disease_Risk',
'Age','Height_cm','Weight_kg','BMI','Systolic_BP',
    'Diastolic_BP','Cholesterol_Total','Cholesterol_HDL',
    'Cholesterol_LDL','Fasting_Blood_Sugar']).values


Yp_test = df_test['Heart_Disease_Risk'].values


# origin dts 

X_train = df_train.drop(columns = ['Heart_Disease_Risk','PC1','PC2']).values

Y_train = df_train['Heart_Disease_Risk'].values

X_test = df_test.drop(columns = ['Heart_Disease_Risk','PC1','PC2']).values
Y_test = df_test['Heart_Disease_Risk'].values



In [50]:
x_p = df.drop(columns = ['Patient_ID','Heart_Disease_Risk',
'Age','Height_cm','Weight_kg','BMI','Systolic_BP',
    'Diastolic_BP','Cholesterol_Total','Cholesterol_HDL',
    'Cholesterol_LDL','Fasting_Blood_Sugar']).values
y_p = df['Heart_Disease_Risk'].values
Xp_train,Xp_test, Yp_train,Yp_test = train_test_split(x_p,y_p,test_size=0.2,random_state=42)


x = df.drop(columns = ['Patient_ID','Heart_Disease_Risk','PC1','PC2']).values
y = df['Heart_Disease_Risk'].values
X_train,X_test, Y_train,Y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [51]:


rd = random_forest(n_tree=15,max_depth=5,min_samp_split = 15) 
rd.fit(X_train,Y_train,feature_names_origin)
predict = rd.predict(X_test)
accuracy = np.mean(predict == Y_test)*100
print(f"origin accuracy: {accuracy}%")



origin accuracy: 71.46666666666667%


In [52]:

rd = random_forest(n_tree=15,max_depth=5,min_samp_split = 15) 
rd.fit(Xp_train,Yp_train,feature_names_pca)
predict = rd.predict(Xp_test)
accuracy = np.mean(predict == Yp_test)*100
print(f"pca accuracy: {accuracy}%")



pca accuracy: 72.5%


cv

In [53]:
def k_fold_cv(X, y, k=3, n_tree=5, max_depth=5, min_samp_split=15):
    np.random.seed(42)
    
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    
    folds = np.array_split(indices, k)
    accuracies = []

    for i in range(k):
        print(f"Fold {i+1}/{k} running...")

        val_idx = folds[i]
        train_idx = np.hstack([folds[j] for j in range(k) if j != i])

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        model = random_forest(
            n_tree=n_tree,
            max_depth=max_depth,
            min_samp_split=min_samp_split
        )

        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        acc = np.mean(preds == y_val) * 100
        accuracies.append(acc)

        print(f"Fold {i+1} accuracy: {acc:.2f}%\n")

    print("CV accuracies:", accuracies)
    print(f"Mean CV accuracy: {np.mean(accuracies):.2f}%")

    return accuracies

In [54]:
cv_acc = k_fold_cv(
    X_train,
    Y_train,
    k=10,              
    n_tree=15,         
    max_depth=5,
    min_samp_split=15
)


Fold 1/10 running...
Fold 1 accuracy: 73.08%

Fold 2/10 running...
Fold 2 accuracy: 73.08%

Fold 3/10 running...
Fold 3 accuracy: 73.50%

Fold 4/10 running...
Fold 4 accuracy: 70.67%

Fold 5/10 running...
Fold 5 accuracy: 72.33%

Fold 6/10 running...
Fold 6 accuracy: 71.67%

Fold 7/10 running...
Fold 7 accuracy: 71.83%

Fold 8/10 running...
Fold 8 accuracy: 70.42%

Fold 9/10 running...
Fold 9 accuracy: 70.83%

Fold 10/10 running...
Fold 10 accuracy: 70.67%

CV accuracies: [np.float64(73.08333333333333), np.float64(73.08333333333333), np.float64(73.5), np.float64(70.66666666666667), np.float64(72.33333333333334), np.float64(71.66666666666667), np.float64(71.83333333333334), np.float64(70.41666666666667), np.float64(70.83333333333334), np.float64(70.66666666666667)]
Mean CV accuracy: 71.81%


In [55]:
cv_acc = k_fold_cv(
    Xp_train,
    Yp_train,
    k=10,              
    n_tree=15,         
    max_depth=5,
    min_samp_split=15
)

Fold 1/10 running...
Fold 1 accuracy: 71.25%

Fold 2/10 running...
Fold 2 accuracy: 73.75%

Fold 3/10 running...
Fold 3 accuracy: 75.50%

Fold 4/10 running...
Fold 4 accuracy: 71.17%

Fold 5/10 running...
Fold 5 accuracy: 70.92%

Fold 6/10 running...
Fold 6 accuracy: 72.08%

Fold 7/10 running...
Fold 7 accuracy: 71.83%

Fold 8/10 running...
Fold 8 accuracy: 69.67%

Fold 9/10 running...
Fold 9 accuracy: 71.33%

Fold 10/10 running...
Fold 10 accuracy: 70.42%

CV accuracies: [np.float64(71.25), np.float64(73.75), np.float64(75.5), np.float64(71.16666666666667), np.float64(70.91666666666666), np.float64(72.08333333333333), np.float64(71.83333333333334), np.float64(69.66666666666667), np.float64(71.33333333333334), np.float64(70.41666666666667)]
Mean CV accuracy: 71.79%


In [56]:


tree_id = 3
rd.trees[tree_id].print_tree()



[Family_History <= 0.5]
  [Stress_Level <= 3.5]
    [Gender <= 0.5]
      [Smoking_Status <= 0.5]
        [Physical_Activity_Level <= 0.5]
          Leaf: 0.0
          Leaf: 0.0
        [Physical_Activity_Level <= 0.5]
          Leaf: 1.0
          Leaf: 1.0
      [Physical_Activity_Level <= 1.5]
        [Physical_Activity_Level <= 0.5]
          Leaf: 0.0
          Leaf: 0.0
        [Sleep_Hours <= 4.5]
          Leaf: 1.0
          Leaf: 0.0
    [Physical_Activity_Level <= 1.5]
      [Smoking_Status <= 0.5]
        [Stress_Level <= 8.5]
          Leaf: 0.0
          Leaf: 0.0
        [Sleep_Hours <= 6.5]
          Leaf: 1.0
          Leaf: 1.0
      [Sleep_Hours <= 8.5]
        [PC2 <= -0.3687672498405496]
          Leaf: 0.0
          Leaf: 0.0
        [Stress_Level <= 9.5]
          Leaf: 0.0
          Leaf: 1.0
  [PC2 <= 0.15850135569526264]
    [Smoking_Status <= 0.5]
      [PC2 <= -0.08922553973177325]
        [Sleep_Hours <= 5.5]
          Leaf: 1.0
          Leaf: 1.0
       